In [4]:
%pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 48.3 MB 535 kB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# 1. TRANSLATION MATRIX
def translation_matrix(tx, ty):
    return np.array([
        [1, 0, tx],
        [0, 1, ty],
        [0, 0, 1]
    ], dtype=np.float32)



# 2. ROTATION MATRIX
def rotation_matrix(angle, cx, cy):
    theta = np.radians(angle)

    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    # Move image center to origin
    T1 = np.array([
        [1, 0, -cx],
        [0, 1, -cy],
        [0, 0, 1]
    ], dtype=np.float32)

    # Rotation matrix
    R = np.array([
        [cos_theta, -sin_theta, 0],
        [sin_theta, cos_theta, 0],
        [0, 0, 1]
    ], dtype=np.float32)

    # Move origin back to image center
    T2 = np.array([
        [1, 0, cx],
        [0, 1, cy],
        [0, 0, 1]
    ], dtype=np.float32)

    return T2 @ R @ T1



# 3. SCALING MATRIX
def scaling_matrix(sx, sy, cx, cy):

    # Move center to origin
    T1 = np.array([
        [1, 0, -cx],
        [0, 1, -cy],
        [0, 0, 1]
    ], dtype=np.float32)

    # Scaling
    S = np.array([
        [sx, 0, 0],
        [0, sy, 0],
        [0, 0, 1]
    ], dtype=np.float32)

    # Move center back
    T2 = np.array([
        [1, 0, cx],
        [0, 1, cy],
        [0, 0, 1]
    ], dtype=np.float32)

    return T2 @ S @ T1


# 4. SHEARING MATRIX
def shearing_matrix(shx, shy, cx, cy):

    # Move center to origin
    T1 = np.array([
        [1, 0, -cx],
        [0, 1, -cy],
        [0, 0, 1]
    ], dtype=np.float32)

    # Shearing
    H = np.array([
        [1, shx, 0],
        [shy, 1, 0],
        [0, 0, 1]
    ], dtype=np.float32)

    # Move center back
    T2 = np.array([
        [1, 0, cx],
        [0, 1, cy],
        [0, 0, 1]
    ], dtype=np.float32)

    return T2 @ H @ T1


# 5. APPLY AFFINE TRANSFORMATION
def apply_affine(image, matrix):

    height, width = image.shape[:2]

    # OpenCV uses a 2 x 3 affine matrix
    affine_matrix = matrix[:2, :]

    transformed_image = cv2.warpAffine(
        image,
        affine_matrix,
        (width, height),
        borderMode=cv2.BORDER_REFLECT
    )

    return transformed_image


# 6. PRINT MATRIX
def print_matrix(name, matrix):

    print("\n" + name)
    print(np.round(matrix, 3))


# 7. SAVE IMAGE
def save_image(folder, filename, image):

    path = os.path.join(folder, filename)

    cv2.imwrite(path, image)

    return path


#MAIN PROGRAM
def main():

    # Input image
    input_path = "input.jpg"

    # Folder for output images
    output_folder = "augmented_images"

    os.makedirs(output_folder, exist_ok=True)

    
    # READ IMAGE
    image = cv2.imread(input_path)

    if image is None:

        print("ERROR: input.jpg was not found.")

        print("\nPlease place an image named 'input.jpg'")
        print("in the same folder as this Python program.")

        return

    # Image dimensions
    height, width = image.shape[:2]

    # Image center
    cx = width / 2
    cy = height / 2


    
    # TRANSFORMATION PARAMETER

    # Translation
    tx = 60
    ty = 40

    # Rotation
    angle = 30

    # Scaling
    sx = 1.25
    sy = 0.80

    # Shearing
    shx = 0.25
    shy = 0.10


    # CREATE MATRICES

    T = translation_matrix(tx, ty)

    R = rotation_matrix(
        angle,
        cx,
        cy
    )

    S = scaling_matrix(
        sx,
        sy,
        cx,
        cy
    )

    H = shearing_matrix(
        shx,
        shy,
        cx,
        cy
    )


    # APPLY TRANSFORMATIONS
    translated = apply_affine(
        image,
        T
    )

    rotated = apply_affine(
        image,
        R
    )

    scaled = apply_affine(
        image,
        S
    )

    sheared = apply_affine(
        image,
        H
    )

    # COMBINED TRANSFORMATIONS
    combined_matrix = T @ R @ S

    combined = apply_affine(
        image,
        combined_matrix
    )

    # DISPLAY MATRICES
    print_matrix(
        "Translation Matrix",
        T
    )

    print_matrix(
        "Rotation Matrix",
        R
    )

    print_matrix(
        "Scaling Matrix",
        S
    )

    print_matrix(
        "Shearing Matrix",
        H
    )

    print_matrix(
        "Combined Transformation Matrix",
        combined_matrix
    )


    # SAVE IMAGES

    save_image(
        output_folder,
        "translated.jpg",
        translated
    )

    save_image(
        output_folder,
        "rotated.jpg",
        rotated
    )

    save_image(
        output_folder,
        "scaled.jpg",
        scaled
    )

    save_image(
        output_folder,
        "sheared.jpg",
        sheared
    )

    save_image(
        output_folder,
        "combined.jpg",
        combined
    )


    # DISPLAY ORIGINAL + AUGMENTED IMAGES

    images = [
        ("Original", image),
        ("Translated", translated),
        ("Rotated", rotated),
        ("Scaled", scaled),
        ("Sheared", sheared),
        ("Combined", combined)
    ]

    plt.figure(figsize=(12, 8))

    for i, (title, img) in enumerate(images):

        plt.subplot(2, 3, i + 1)

        # OpenCV uses BGR
        # Matplotlib uses RGB
        img_rgb = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        plt.imshow(img_rgb)

        plt.title(title)

        plt.axis("off")


    plt.tight_layout()

    # Save comparison
    plt.savefig(
        os.path.join(
            output_folder,
            "comparison.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    # Show images
    plt.show()


    print("\n-----------------------------------")
    print("DATA AUGMENTATION COMPLETED")
    print("-----------------------------------")

    print("\nImages saved in:")
    print(output_folder)


if __name__ == "__main__":
    main()

ERROR: input.jpg was not found.

Please place an image named 'input.jpg'
in the same folder as this Python program.


[ WARN:0@3.494] global loadsave.cpp:278 findDecoder imread_('input.jpg'): can't open/read file: check file path/integrity
